In [ ]:
import os
from glob import glob
import time

from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support.expected_conditions import presence_of_all_elements_located

In [ ]:
data = pd.read_excel(
    os.path.expanduser('~/Downloads/esklp_20241107_excel_00001/esklp_smnn_20241107.xlsx'), sheet_name=1,
) # pip install openpyxl
data.head()

/home/laks/310env/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Стандартизованное МНН,Код узла СМНН,Код ОКПД 2,Стандартизованная лекарственная форма,Стандартизованная дозировка,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Единица измерения лекарственного препарата,...,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Список нормализованных МНН для узла СМНН,Список нормализованных лекарственных форм и дозировок для узла СМНН,КЛП,Некорректные данные
0,NaN,NaN,NaN,NaN,Кол-во,Единица измерения,NaN,NaN,Описание,NaN,...,Диапазон II количества единиц измерения лекарс...,NaN,Значение цены III,Диапазон III количества единиц измерения лекар...,NaN,Показатель среднеквадратичного отклонения (σ),NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,Наименование,Код ОКЕИ,Наименование единицы из ОКЕИ,NaN,Наименование,...,Минимальное,Максимальное,NaN,Минимальное,Максимальное,NaN,NaN,NaN,Ссылка,NaN
2,1,2,3,4,5,6,7,8,9,10,...,28,29,30,31,32,33,34,35,36,37
3,1-(4-БРОМФЕНИЛ)ВИОЛУРОВАЯ КИСЛОТА,21.20.10.114-000050-1-00082-0000000000000,21.20.10.114,КАПСУЛЫ,500,мг,161,мг,500 мг,шт. (капсула),...,NaN,NaN,NaN,NaN,NaN,NaN,1-(4-БРОМФЕНИЛ)ВИОЛУРОВАЯ КИСЛОТА,КАПСУЛЫ (500 мг),esklp_klp_20241107_00000.xlsx,NaN
4,1-[2-(1-МЕТИЛИМИДАЗОЛ-4-ИЛ)-ЭТИЛ]ПЕРГИДРОАЗИН-...,21.20.10.214-000057-1-00001-0000000000000,21.20.10.214,ТАБЛЕТКИ,100,мг,161,мг,100 мг,шт. (таблетка),...,NaN,NaN,NaN,NaN,NaN,NaN,1-[2-(1-МЕТИЛИМИДАЗОЛ-4-ИЛ)-ЭТИЛ]ПЕРГИДРОАЗИН-...,ТАБЛЕТКИ (100 мг),esklp_klp_20241107_00000.xlsx,NaN


In [ ]:
data.columns

Index(['Стандартизованное МНН', 'Код узла СМНН', 'Код ОКПД 2',
       'Стандартизованная лекарственная форма', 'Стандартизованная дозировка',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8',
       'Единица измерения лекарственного препарата', 'Unnamed: 10',
       'Unnamed: 11', 'Наименование ФТГ', 'АТХ', 'Unnamed: 14', 'ЖНВЛП',
       'Наличие в лекарственном препарате наркотических средств, психотропных веществ и их прекурсоров',
       'Период действия узла СМНН', 'Unnamed: 18', 'Дата изменения записи',
       'Референтные цены (приведенные значения цен, в соответствии с письмом Минздрава России, являются средневзвешенные по Российской Федерации и справочными)',
       'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24',
       'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28',
       'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32',
       'Список нормализованных МНН для узла СМНН',
       'Список нормализованных лекарственных форм и дозиров

In [ ]:
data = data[~data['Наименование ФТГ'].isna()][3:]
data.head()

,Стандартизованное МНН,Код узла СМНН,Код ОКПД 2,Стандартизованная лекарственная форма,Стандартизованная дозировка,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Единица измерения лекарственного препарата,...,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Список нормализованных МНН для узла СМНН,Список нормализованных лекарственных форм и дозировок для узла СМНН,КЛП,Некорректные данные
5,"2,8-ДИМЕТИЛ-5-(2-ФЕНИЛЭТИЛ)-2,3,4,5-ТЕТРАГИДРО...",21.20.10.235-000143-1-00002-0000000000000,21.20.10.235,"ТАБЛЕТКИ, ПОКРЫТЫЕ ОБОЛОЧКОЙ",20,мг,161,мг,20 мг,шт. (таблетка),...,NaN,NaN,NaN,NaN,NaN,NaN,"2,8-ДИМЕТИЛ-5-(2-ФЕНИЛЭТИЛ)-2,3,4,5-ТЕТРАГИДРО...",ТАБЛЕТКИ ПОКРЫТЫЕ ПЛЕНОЧНОЙ ОБОЛОЧКОЙ (20 мг),esklp_klp_20241107_00000.xlsx,NaN
6,4-НИТРО-N-[(1RS)-1-(4-ФТОРФЕНИЛ)-2-(1-ЭТИЛПИПЕ...,21.20.10.141-000047-1-00045-0000000000000,21.20.10.141,КОНЦЕНТРАТ ДЛЯ ПРИГОТОВЛЕНИЯ РАСТВОРА ДЛЯ ВНУТ...,1,мг/мл,876,усл. ед,1 мг/мл,мл,...,NaN,NaN,NaN,NaN,NaN,NaN,4-НИТРО-N-[(1RS)-1-(4-ФТОРФЕНИЛ)-2-(1-ЭТИЛПИПЕ...,КОНЦЕНТРАТ ДЛЯ ПРИГОТОВЛЕНИЯ РАСТВОРА ДЛЯ ВНУТ...,esklp_klp_20241107_00000.xlsx,NaN
7,"7,9-ДИБРОМ-2Н-[1]БЕНЗОПИРАНО[2,3-D]ПИРИМИДИН-2...",21.20.10.191-000129-1-00045-0000000000000,21.20.10.191,КАПСУЛЫ,250,мг,161,мг,250 мг,шт. (капсула),...,NaN,NaN,NaN,NaN,NaN,NaN,"7,9-ДИБРОМ-2Н-[1]БЕНЗОПИРАНО[2,3-D]ПИРИМИДИН-2...",КАПСУЛЫ (250 мг),esklp_klp_20241107_00000.xlsx,NaN
8,ARANEUS DIADEMATUS+HEKLA-LAVA+KALIUM JODATUM+А...,21.20.10.251-000777-1-00005-0000000000000,21.20.10.251,ТАБЛЕТКИ ГОМЕОПАТИЧЕСКИЕ,1,~,876,усл. ед,~,шт. (таблетка),...,NaN,NaN,NaN,NaN,NaN,NaN,~,ТАБЛЕТКИ ДЛЯ РАССАСЫВАНИЯ ГОМЕОПАТИЧЕСКИЕ (НЕ ...,esklp_klp_20241107_00000.xlsx,NaN
9,CHINA RUBRA+АДРЕНАЛИНУМ+ВИНКА МИНОР+ГАМАМЕЛИС ...,21.20.10.251-000807-1-00004-0000000000000,21.20.10.251,КАПЛИ ГОМЕОПАТИЧЕСКИЕ,1,~,876,усл. ед,~,мл,...,NaN,NaN,NaN,NaN,NaN,NaN,~,КАПЛИ ДЛЯ ПРИЕМА ВНУТРЬ ГОМЕОПАТИЧЕСКИЕ (НЕ УК...,esklp_klp_20241107_00000.xlsx,NaN


In [ ]:
data = data[data['Наименование ФТГ'].str.contains('противоопухолевое')]

In [ ]:
downloaded = pd.read_csv('Instructions/instr_metadata.tsv', sep='\t',header=None)
downloaded.tail()
# метаданные всех инструкций, которые мы выкачали
# объяснение метаданных:
# 0 - Стандартизованное МНН, оно же действующее вещество
# 1 - торговое наименование
# 2 - ID лекарства в ГРЛС, из которого можно составить адрес его странички
# 3 - путь до PDF с инструкцией. Пользуясь им, можно найти файл инструкции в выгрузке ({uuid}.pdf)
# записи могут дублироваться, если одна и та же инструкция скачалась > 1 раза (во время дебага например)

,0,1,2,3
565,Этопозид,Этопозид-Тева,28404765-6f64-49da-94a4-4079fe7ab37d,https://grls.rosminzdrav.ru/InstrImg/2023/11/3...
566,Этопозид,Фитозид,eac03547-ea20-413a-b976-9151deb254c4,https://grls.rosminzdrav.ru/InstrImg/2022/01/1...
567,Этопозид,Ластет,5c2006a5-ffd6-487e-8330-2f2ab2619c46,https://grls.rosminzdrav.ru/InstrImg/2015/10/2...
568,Этопозид,Ластет,82190c01-9445-4799-a905-c166ee76a7bc,https://grls.rosminzdrav.ru/InstrImg/2020/05/2...
569,Этопозид,Этопозид-Эбеве®,8fdf4e0b-077d-4aef-9bb0-b99b749b1218,https://grls.rosminzdrav.ru/InstrImg/2021/10/2...


Открываем файл с метаданными в режиме "дописывания" (загрузка иногда прерывалась из-за каптчи. Каптча у них древняя, не было уверенности, что готовые библиотеки для решения reCaptcha подойдут)

In [ ]:
f = open('instr_metadata.tsv', 'a', encoding='utf8')

In [ ]:
options = webdriver.FirefoxOptions()
driver = webdriver.Firefox(options=options)
# запускаем Фаерфокс в режиме бота только один раз. В настройках выбираем действия с pdf - сохранить на диск.
# (это должно было быть возможно автоматизировать через FirefoxProfile, но не сработало)
# так же можно выбрать папку, куда сохранять (по умолчанию это обычная папка Downloads, куда падают все загрузки
# из браузеров)

In [ ]:
for key in data['Стандартизованное МНН'].unique():
    key = key.strip()
    key = key[0] + key[1:].lower()
    if (key not in downloaded[0].values) and (key not in {
        'Арглабин', 'Белок теплового шока gp-96', 'Гефитиниб', 'Коикса семян масло'}):
        # у этих веществ нет лекарств с инструкциями
        print(key)
        wait = WebDriverWait(driver, 10)
        driver.get("https://grls.rosminzdrav.ru/GRLS.aspx")
        wait = WebDriverWait(driver, 10)
        driver.find_element(By.ID, 'ctl00_plate_txtMNN').send_keys(key)
        wait = WebDriverWait(driver, 10)
        driver.find_element(By.ID, 'ctl00_plate_bSeek').click()
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        els = soup.find_all(class_='hi_sys poi')
        for el in els:
            time.sleep(5)
            wait = WebDriverWait(driver, 10)
            onclick = el['onclick'].split("'")[1]
            filename = onclick + '.pdf'

            driver.get(f"https://grls.rosminzdrav.ru/Grls_View_v2.aspx?routingGuid={onclick}")
            time.sleep(10)
            name = driver.find_element(By.ID, 'ctl00_plate_TradeNmR').get_attribute('value')
            print(name)

            driver.find_element(By.ID, 'instructionsCaller').click()
            time.sleep(10)
            try:
                iframe = driver.find_element(By.TAG_NAME, 'iframe')
            except Exception as e:
                # bad practice. По-хорошему надо было бы найти, откуда селениум берет NoSuchElementError
                # и исключать только её
                # это обработка случая, когда инструкции нет
                print(e)
                continue
            path = iframe.get_attribute('src')
            driver.switch_to.frame(iframe)

            f.write(key + '\t' + name + '\t' + onclick + "\t" + path +'\n')


Лизомустин
Ралтитрексед
Тегафур+урацил
Фактор некроза опухоли альфа-1 [тимозин рекомбинантный]
Цитарабин
ЦИТАРАБИН-РУС
Алексан®
Цитарабин-ЛЭНС®
Цитарабин
Цитозар®
Эверолимус
ЭВЕРОЛИМУС
ЭВЕРОЛИМУС
ЭВЕЛИРОК®
Афинитор®
Афинитор®
СЕЙВНОВИ®
Эверолимус-АМЕДАРТ
Сертикан®
Алвида®
ЭВЕРОЛИМУС-ПРОМОМЕД
Эксеместан
Эксеместан-ТЛ
Аромазин®
Элотузумаб
Эмплисити®
Эпирубицин
Эпирубицин-Рус
Эпирубицин-РОНЦ®
Эпирубицин-Келун-Казфарм
Эпирубицин
Фарморубицин® быстрорастворимый
Эрлотиниб
Эрлатера
ЭРЛОТИНИБ-ТЛ
Эрлотиниб
Эрлотиниб
Эрлотиниб-натив
Эрлотиниб
Этопозид
Этопозид-Рус
Этопозид-ЛЭНС®
Фитозид
Филотид
Этопозид
Этопозид-Тева
Фитозид
Ластет
Ластет
Этопозид-Эбеве®


In [ ]:
f.close() # не забудь закрыть после того, как предыдущая ячейка упала с каптчей, иначе не запишется

In [ ]:
# remove duplicated files created while debugging

for path in glob('Instructions/*).pdf'):
    print(path)
    os.remove(path)

Instructions/a967d2ef-a0f1-4371-b71d-0c86e79df628(1).pdf
Instructions/91cb1f69-3eab-424c-94e3-58cef7a03ba0(1).pdf
Instructions/09dbf80f-3cfe-4a2c-82c4-7683ab829172(1).pdf
Instructions/3baf33a6-4020-4d89-91e6-fac72f9bd0fd(1).pdf
Instructions/c6a71699-819e-40a6-a9d3-1a383fa5021f(1).pdf
Instructions/4136123e-fb3f-46a4-9def-aa1517f4d70a(1).pdf
Instructions/5c34e22a-7052-4213-8d64-a0400017807e(1).pdf
Instructions/8709d2e9-8a9f-4a27-a2c8-2258352d112c(1).pdf
Instructions/5a1ebe55-1f7b-443e-a2f6-46d77bd08074(1).pdf
Instructions/7dd7abb0-5932-4476-9a31-5375c903d666(1).pdf
Instructions/d48ef54d-05ce-4a24-8ad8-bcd344ce21fc(1).pdf
Instructions/4a950323-f8c2-4292-9858-59b3591c9b44(2).pdf
Instructions/237cf73e-af1e-49ee-af73-7a3f121608f2(1).pdf
Instructions/58ac3835-4b49-4ba5-bd4e-5c1e414ee154(1).pdf
Instructions/b355dfd5-ae69-4d24-846d-0f82adfb0fc6(1).pdf
Instructions/4a950323-f8c2-4292-9858-59b3591c9b44(1).pdf
Instructions/af2607fd-1618-4d65-aeab-d73b6ee86cf2(1).pdf
Instructions/44863a63-8baf-446a